In [2]:
import math
from typing import Literal

import dnnlpy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

print("PyTorch version: ", torch.__version__)


PyTorch version:  2.13.0+cpu


In [3]:
"""
为什么要减低图片的分辨率
    - 降低后面的计算量
    - 可以提取浅层信息
    具体表现
        -  空间尺逐渐减小
        - 通道数逐渐增加
"""


def conv_macs(**kwargs: dict[str, str]) -> Tensor:
    return math.prod(kwargs.values())


full_resolution = conv_macs(
    batch_size=4,
    in_channels=64,
    out_channels=128,
    image_height=56,
    image_width=56,
    kernel_size=3,
)

half_resolution = conv_macs(
    batch_size=1,
    in_channels=64,
    out_channels=128,
    image_height=28,
    image_width=28,
    kernel_size=3,
)

print(f"56X56 features map: {full_resolution} MACs")
print(f"28X28 features map: {half_resolution} MACs")
print(f"Ratio:{half_resolution / full_resolution:.4f}")

56X56 features map: 308281344 MACs
28X28 features map: 19267584 MACs
Ratio:0.0625


In [4]:
"""
最大池化层:保留局部区域中的强响应
    每个局部窗口中保留最大值
"""
x = torch.arange(1, 17, dtype=torch.float32).view(1, 1, 4, 4)
y = F.max_pool2d(x, kernel_size=2, stride=2)

print("Input:", x[0, 0], sep='\n')
print("max pooled Output:", y[0, 0], sep='\n')

left = torch.tensor([[[[0.0, 5.0], [0.0, 0.0]]]])
right = torch.tensor([[[[0.0, 0.0], [5.0, 0.0]]]])

left = F.max_pool2d(left, kernel_size=2, stride=2)
right = F.max_pool2d(right, kernel_size=2, stride=2)

print("left:", left[0, 0], sep='\n')
print("right:", right[0, 0], sep='\n')


Input:
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.],
        [13., 14., 15., 16.]])
max pooled Output:
tensor([[ 6.,  8.],
        [14., 16.]])
left:
tensor([[5.]])
right:
tensor([[5.]])


In [5]:
"""
平均池化层:汇总局区域的整体响应
"""
y = F.avg_pool2d(x, kernel_size=2, stride=2)
print("Average pooled output:", y[0, 0], sep='\n')

feature = torch.tensor([[[[1.0, 1.0], [1.0, 9.0]]]])

max_value = F.max_pool2d(feature, kernel_size=2, stride=2)
avg_value = F.avg_pool2d(feature, kernel_size=2, stride=2)

print("Max pooled output:", max_value.item(), sep='\n')
print("Avg pooled output:", avg_value.item(), sep='\n')


Average pooled output:
tensor([[ 3.5000,  5.5000],
        [11.5000, 13.5000]])
Max pooled output:
9.0
Avg pooled output:
3.0


In [6]:
"""
池化层的输出尺寸
"""

x = torch.randn(4, 16, 32, 32)
y = F.max_pool2d(y, kernel_size=2, stride=2)
print('Input  shape:', x.shape, sep='\n')
print("output shape:", y.shape, sep='\n')


Input  shape:
torch.Size([4, 16, 32, 32])
output shape:
torch.Size([1, 1, 1, 1])


In [9]:
"""
    从零实现二维池化
        池化只可以改变H和W,通道数不会改变
"""


def pool2d(
        x: Tensor,
        kernel_size: int,
        stride: int | None = None,
        mode: Literal["avg", "max"] = "max",
) -> Tensor:
    if x.ndim != 4:
        raise AssertionError("Input tensor must be 4D")
    if kernel_size <= 0:
        raise AssertionError("kernel_size must be greater than 0")
    if stride is None:
        stride = kernel_size
    if stride <= 0:
        raise AssertionError("Stride must be greater than 0")

    batch_size, in_channels, input_h, input_w = x.shape
    output_h = (input_h - kernel_size) // stride + 1
    output_w = (input_w - kernel_size) // stride + 1

    if output_h <= 0 or output_w <= 0:
        raise AssertionError("Output shape must be greater than 0")
    output = x.new_zeros(batch_size, in_channels, output_h, output_w)
    for i in range(output_h):
        h_start = i * stride
        h_end = h_start + kernel_size
        for j in range(output_w):
            w_start = j * stride
            w_end = w_start + kernel_size
            window = x[:, :, h_start:h_end, w_start:w_end]

            if mode == "max":
                output[:, :, i, j] = window.amax(dim=(-2, -1))
            elif mode == "avg":
                output[:, :, i, j] = window.mean(dim=(-2, -1))
            else:
                raise AssertionError("Mode must be 'max' or 'avg'")
    return output


x = torch.randn(2, 3, 7, 8)

max_actual = pool2d(x, kernel_size=3, stride=2, mode="max")
max_expected = F.max_pool2d(x, kernel_size=3, stride=2)

avg_actual = pool2d(x, kernel_size=3, stride=2, mode="avg")
avg_expected = F.avg_pool2d(x, kernel_size=3, stride=2)

max_flag = torch.allclose(max_actual, max_actual)
avg_flag = torch.allclose(avg_actual, avg_expected)
print("Is max  pooling matches?", max_flag)
print("Is avg  pooling matches?", avg_flag)

Is max  pooling matches? True
Is avg  pooling matches? True


In [10]:
negative = torch.tensor([[[[-5.0, -4.0], [-3.0, -2.0]]]])
pooled = F.max_pool2d(negative, kernel_size=2, stride=1, padding=1)
print(pooled[0, 0])

tensor([[-5., -4., -4.],
        [-3., -2., -2.],
        [-3., -2., -2.]])


In [11]:
x = torch.ones(1, 1, 2, 2)

include_pad = F.avg_pool2d(
    x, kernel_size=2, stride=1, padding=1,
    count_include_pad=True
)

exclude_pad = F.avg_pool2d(
    x, kernel_size=2, stride=1, padding=1,
    count_include_pad=False
)

print("Include padding in driver:", include_pad[0][0], sep='\n')
print("Exclude padding in driver:", exclude_pad[0][0], sep='\n')

Include padding in driver:
tensor([[0.2500, 0.5000, 0.2500],
        [0.5000, 1.0000, 0.5000],
        [0.2500, 0.5000, 0.2500]])
Exclude padding in driver:
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])


In [13]:
"""
池化层如何反向传播
    最大池化的梯度较为稀疏
    平均池化会让每个局部位置都可以接受到梯度
"""

x = torch.tensor([[[[1.0, 3.0], [2.0, 4.0]]]], requires_grad=True)
y = F.max_pool2d(x, kernel_size=2)
y.backward()

print("max pooled value:", y.item(), sep='\n')
print("Gradient of max:", x.grad[0][0], sep='\n')

y = F.avg_pool2d(x, kernel_size=2)
y.backward()

print("avg pooled value:", y.item(), sep='\n')
print("Gradient of avg:", x.grad[0][0], sep='\n')

max pooled value:
4.0
Gradient of max:
tensor([[0., 0.],
        [0., 1.]])
avg pooled value:
2.5
Gradient of avg:
tensor([[0.2500, 0.2500],
        [0.2500, 1.2500]])


In [14]:
"""
Adaptive Pooling: 直接决定输出的大小
"""

adaptive_pool = nn.AdaptiveAvgPool2d(1)

for shape in [(2, 64, 7, 7), (2, 64, 14, 14), (2, 4, 10, 16)]:
    x = torch.randn(shape)
    y = adaptive_pool(x)
    print('shape:', x.shape, sep='\n')

shape:
torch.Size([2, 64, 7, 7])
shape:
torch.Size([2, 64, 14, 14])
shape:
torch.Size([2, 4, 10, 16])


In [16]:
x = torch.randn(2, 3, 5, 7)

actual = x.mean(dim=(-2, -1), keepdim=True)
excepted = F.adaptive_avg_pool2d(x, output_size=(1, 1))

flag = torch.allclose(actual, excepted)
print("Is global  pooling  matches  manual mean?", flag)

channels = 512
height = width = 7
num_classes = 1000

flatten_head = nn.Linear(channels * height * width, num_classes)
gap_head = nn.Linear(channels, num_classes)

flatten_params = dnnlpy.count_params(flatten_head)
gap_params = dnnlpy.count_params(gap_head)

print(f"Flatten params: {flatten_params}\nGap params: {gap_params}\n")

Is global  pooling  matches  manual mean? True
Flatten params: 25089000
Gap params: 513000



In [18]:
"""
pooling与strided  Convolution
    最大池化
        - 没有可以学习的参数
        - 每个通道独立处理
        - 通常不改变通道数
        - 使用固定的最大值规则全局信息
        
    Strided Convolution
        - 包含可学习权重
        - 可以混合输入通道
        - 可以改变空间尺寸和通道数
        - 下采样规则由数据学习得到
"""

x = torch.randn(2, 16, 32, 32)

conv = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)
pool = nn.MaxPool2d(kernel_size=2)

conv_output = conv(x)
pool_output = pool(x)

print('MaxPool  output:', pool_output.shape, sep='\n')
print('Strided Convolution output:', conv_output.shape, sep='\n')

MaxPool  output:
torch.Size([2, 16, 16, 16])
Strided Convolution output:
torch.Size([2, 32, 16, 16])
